# Canada's Education Census Data Explorer

## Introduction
This project involves interacting with Canada's data on the number of people with education qualifications such as bachelor’s degrees within different regions in Canada. The types of regions, such as census divisions, and different levels of qualifications can be interactively selected and displayed to the user to explore a corresponding map. This map will be a choropleth with a corresponding colour bar by dividing Canada into the selected boundary to segregate by. The user can move and zoom around in the map to explore the data in a simple and active way.


In [1]:
from ipyleaflet import *
import ipywidgets as widgets
import geopandas as gpd
import pandas as pd
import os.path
import numpy as np
from branca.colormap import linear

#To have the map load quickly for a smaller area, set debug to True
debug=True

In [2]:
if debug:
    #zooms in on Ontario's lower mainland
    defaultLong = -76.71
    defaultLat = 46.00
    defaultZoom = 6
else:
    #default area is the center of Canada 
    defaultLong = -103.11
    defaultLat = 62.13
    defaultZoom = 4


m = Map(center=(defaultLat,defaultLong), zoom=defaultZoom)

## Data Sources
The data used in this project was all sourced from Canadian government's official website. This consisted of utilizing shapefile boundary files for Canada's provinces and territories, census divisions and census subdivisions alongside census data about education certificates per region. This data was accessible without any application or sign-up process, which made obtaining it was not cumbersome in a personal privacy perspective. However, navigation through the site was not as trivial due to how many options there were for census data. Although finding specific dataset such as education did not involve too much effort, finding data in a format that was able to be combined with the shapfiles proved more challenging than initially thought.

The census data came in the form of CSV files, where it needed to be parsed for information pertaining to university education certificates only. This was initially stored in a very strange and inefficient way. All certificates, whether that is a university accredited one or not, was push into a single column. For the purposes of this notebook, all the values in the columns should have each had its own column, this meaning the column should have been a row. Another way the census data was formatted strangely was regarding the repeated DGUIDs, which are the unique identifiers per region in the shapefiles. The logical way to store this data would be to have one row per DGUID with its corresponding counts for all certificates. 

## Methods
Due to the bizarre formatting of the csv census data as explained above, heavy manipulation of the table was required. Firstly, a simple query was used to obtain only relevant columns to the project. These columns where the education certificates and total number of people with a certificate, with the inclusion of the DGUID column to combine the data with the boundary files. This does not solve the formatting but makes creating a new table formatted correctly easier later in the notebook. 

In [3]:
educ = pd.read_csv("education.csv")
print("Done reading CSV's")
#only select the useful columns of the CSV
#In a general sense, the 3 columns are DGUID, catagories, and counts
educ = educ[["DGUID","Highest certificate, diploma or degree (16)","Place of work status (7):Total - Place of work status[1]"]]

print("Done extracting CSV's useful data")

Done reading CSV's
Done extracting CSV's useful data


In [4]:
#The resulting table. This has now only has necessary data for the interactive map, but the fortmatting still requires manipulation depending on which boundary and education certificate is chosen
display(educ)

,DGUID,"Highest certificate, diploma or degree (16)",Place of work status (7):Total - Place of work status[1]
0,2021A000011124,"Total - Highest certificate, diploma or degree",17321700.0
1,2021A000011124,"No certificate, diploma or degree",1477990.0
2,2021A000011124,High (secondary) school diploma or equivalency...,4124800.0
3,2021A000011124,"Postsecondary certificate, diploma or degree",11718910.0
4,2021A000011124,Postsecondary certificate or diploma below bac...,5886090.0
...,...,...,...
262459,2021A00056208098,Bachelor's degree,NaN
262460,2021A00056208098,University certificate or diploma above bachel...,NaN
262461,2021A00056208098,"Degree in medicine, dentistry, veterinary medi...",NaN
262462,2021A00056208098,Master's degree,NaN


This interactive notebook was possible by using a Leaflet map and whilst shapefiles can be added to this digital map, GeoJSON files seemed to be far simpler, at least for this Python library.  The initial boundary files for Canada were obtained as a shapefile, so conversion was needed. Originally, the project manually converted the shapefiles to GeoJSON files. This, however, took significant time and was unable to convert the files into a usable format. To combat this, the external application QGIS was used to convert the files.

 


By analyzing the time and output from QGIS, the boundary files were significantly larger than what Python and Leaflet could handle. Many reiterations of simplification on all the boundaries files were done in QGIS to reduce the file size. The census subdivisions file in particular was difficult to reduce the file size without destroying significant geometry. Using these simplified layers in the notebook, the loading time for these boundaries became rather long, unreasonably long for census subdivisions. To fix this aspect for testing, a debug option was programmed to only display a small section of the shapefiles. This could also be used to enhance user experience, although defeats the purpose of exploring.

Putting both datasets together, a merge between the GeoJSON boundary files and CSV filtered census data was done. Both the boundaries and census data were mapped based on their common DGUID columns.

In [5]:
#read the geojson files and append the opened CSV data to it
def mergeData(geojsonName):
    file = gpd.read_file(geojsonName)
    return file.merge(educ, on="DGUID")
if debug:
    #a smaller region on Ontario's lower mainland
    provTerrSimp_merged = mergeData("provterr_clip.geojson")
    cenDivSimp_merge = mergeData("divisions_clip.geojson")
    cenSubDivSimp_merge = mergeData("subdivisions_clip.geojson")
else:
    #the entirety of Canada
    provTerrSimp_merged = mergeData("provterr.geojson")
    cenDivSimp_merge = mergeData("divisions.geojson")
    cenSubDivSimp_merge = mergeData("subdivisions.geojson")
print("Done merging data")

Done merging data


ERROR 1: PROJ: proj_create_from_database: Open of /anaconda3/envs/gis-base/share/proj failed


After preparing the data in a usable form, two drop down widgets are displayed for the user to select between which boundary file to add to the leaflet map and what university certificate to colour code each feature. The selections the user makes only updates the map when the "Apply Data" button is pressed. When this happens, all existing boundary layers and colour bars are removed from the leaflet map. After setting an empty canvas, the boundary text selection is keyed to its corresponding GeoJSON file. This file contains the table of census data, but the university certificates are stored as a singular column instead of separately. The easiest way found during this study was to select all rows in the certificates column that match what the user desired and create a new attribute table with that column having all the same value applied to the GeoJSON.

Once the GeoJSON is formatted perfectly, the final step is to add the GeoJSON to the leaflet map as a choropleth map, this being a map with features categorized and coloured based on a value per feature. Due to the above method of selecting only certain rows from the original, the choropleth class was unable to display the GeoJSON due to indices of each row being scattered. This was solved by adding an "id" row the the GeoJSON attributes for the choropleth to properly index.

Finally, the layer is then added to the leaflet map with a fitting color bar that the user can zoom and pan around in. The process described above will repeat every time the user hits the "Apply Data" button when the user wants to change what combination of data they want to display.

In [6]:
#This function updates the map when the apply button is clicked
def drawMap(caller):
    with enterOutput:
        #remove existing layers so we only display tyhe one we add later on
        for i in range(1, len(m.layers)):
            m.remove_layer(m.layers[1])

        #remove the colourbar associated with the past layer
        #note how we do not remove the 0th control: that control is the zoom in and out feature
        for i in range(1, len(m.controls)):
            m.remove_control(m.controls[i])
        
        #get the correct geojson file depending on what boundary was chosen with data on the eductation level chosen
        attributesMain = categoryKeyToVal(divKeyToVal())

        #this is done so the choropleth has an id column to match on
        attributesMain.insert(2, 'id', attributesMain.index)

        #add the choropleth layer to the map
        m.add_layer(
            Choropleth(
                    geo_data=json.loads(attributesMain.to_crs(4326).to_json()),
                    #The two columns the choropleth will work with, one being the identifie, the other being the numeric value
                    choro_data=dict(zip(map(str, attributesMain["id"].tolist()),attributesMain["Place of work status (7):Total - Place of work status[1]"].tolist())),
                    colormap=linear.RdYlBu_09,
                    style={'fillOpacity': 1.0, "color":"black"},
                    key_on='id'
            )
        )     
        
        #get the name of the education level we are sorting on. Note that all values in the "Highest certificate, diploma or degree (16)" column are the same
        name  = attributesMain["Highest certificate, diploma or degree (16)"].tolist()[0]
        #add a colourbar to the map
        colormap_control = ColormapControl(
            caption= "Number of people with a(n) "+name,
            colormap=linear.RdYlBu_09,
            value_min=round(min(attributesMain["Place of work status (7):Total - Place of work status[1]"])),
            value_max=round(max(attributesMain["Place of work status (7):Total - Place of work status[1]"])),
            position='topright',
            transparent_bg=True
        )
        m.add(colormap_control)

#Gets the user's chosen boundary geojson file
def divKeyToVal():
    inputDiv = selectDivision.value
    if inputDiv == 'Provinces/Territories':
        return provTerrSimp_merged
    elif inputDiv == 'Census Divisions':
        return cenDivSimp_merge
    elif inputDiv == 'Census Subdivisions':
        return cenSubDivSimp_merge
    return 0

#Adds to a given boundary geojson file to have the user requested education catagory data
def categoryKeyToVal(df):
    input = selectCategory.value
    if input == 1:
        return df[df["Highest certificate, diploma or degree (16)"]  == "High (secondary) school diploma or equivalency certificate"]
    elif input == 2:
        return df[df["Highest certificate, diploma or degree (16)"]  == "Bachelor's degree"]
    elif input == 3:
        return df[df["Highest certificate, diploma or degree (16)"]  == "Master's degree"]
    elif input == 4:
        return df[df["Highest certificate, diploma or degree (16)"]  == "Earned doctorate"]
    return df

#Drop down selector for boundaries
selectDivision = widgets.Dropdown(
    options=['Provinces/Territories', 'Census Divisions', 'Census Subdivisions'],
    value = 'Provinces/Territories',
    description='Boundary',
)
#Drop down selector for eductation catagory
selectCategory = widgets.Dropdown(
    options=[("High School Diploma",1),("Bachelor's Degree",2),("Master's Degree",3),("Doctorate",4)],
    value = 2,
    description='Education Level',
)
#Button to apply selections from dropdowns
enterButton = widgets.Button(
    description='Apply Data',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Applies the selected data to the map',
    icon='check'
)
m.layout.height = "1000px"

# Results/How to Use
### -select a boundary to divide Canada by
### -select a university certificate to categorize the boundaries by
### -hit apply to add your custom layer to the map
### -explore!

A color bar will be displayed indicating the quantity on people with the given certificate per region. Note that a black region means no value.

In [7]:
#Display dropdown selections
display(selectDivision)
display(selectCategory)
#Set up and display the apply button
enterOutput = widgets.Output(layout={'border': '1px solid white'})
display(enterOutput)
display(enterButton)
enterButton.on_click(drawMap)

m

Dropdown(description='Boundary', options=('Provinces/Territories', 'Census Divisions', 'Census Subdivisions'),…

Dropdown(description='Education Level', index=1, options=(('High School Diploma', 1), ("Bachelor's Degree", 2)…

Output(layout=Layout(border_bottom='1px solid white', border_left='1px solid white', border_right='1px solid w…

Button(button_style='info', description='Apply Data', icon='check', style=ButtonStyle(), tooltip='Applies the …

Map(center=[46.0, -76.71], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

# Conclusion
While doing this project, there were three main issues that took significantly more time than expect, these issues also being completely unexpected. The first one that came up that baffled me was when GeoPandas was unable to read the original boundary shapefiles. I believe the root of the issue came from the projection not being matched to leaflet's expected projection. This was rather frustrating because there was no error message or any output; it was as if the layer did not even attempt to add itself. The content of the course showed us how to a GeoJSON file instead of a shapefile, and I originally attempted to convert them using Python. This also secretly failed in the end, although the projection once again might have been to blame.

The second issue pertained to the format of the census data, specifically how all certificates were store in a single column rather than each a separate row. As shown in the methods explained above, the majority of the project surrounded getting this data in a usable format for the project. There was a far more simple way to turn a column into a row, but all attempts to do so not only were too complex in relation to my knowledge but also failed in the end. Studying more on how to manipulate CSV and other row-column file types would be most beneficial to simplify and clean the code.

The final problem was sourced from the second, this being the trouble adding the modified GeoJSON file as a choropleth layer. The parameters for this class or function were very unclear as resources online were fairly limited. The issue in the end came from the way a new attribute table was made and applied to the GeoJSON file due to inaccurate indices. The solution of adding an "id" column worked, but I would have liked to know more about why the indices were scattered. Understanding more about the different types of layer, such as a choropleth layer, would have been helpful for understanding and code cleanness.

There was one other issue that did not necessarily stop the progress of programming the notebook but severely hindered interactivity and testing, this being the length of time needed to open and add the boundary files. Due to the sheer size of the country, the files were massive and heavy simplification was needed to load at a somewhat reasonable rate. This, however, could only be simplified so much before the geometry would break. Even after simplifying as much as possible, the boundary files take several seconds to several minutes to load. This is particularly true for the census subdivisions for all of Canada. This took sometimes 5 minutes to apply to the map, which is completely pointless in terms of an interactive tool. A faster way to query and load the data is needed to make this project functional for real-time interaction. For testing purposes, a debug mode using a small portion of the file was used during development. This gave very quick results due to a massive drop in file size.

# Bibliography
https://www12.statcan.gc.ca/census-recensement/2021/geo/sip-pis/boundary-limites/index2021-eng.cfm?year=21

https://www150.statcan.gc.ca/n1/en/type/data?MM=1

https://github.com/jupyter-widgets/ipyleaflet/blob/master/examples/Choropleth.ipynb

https://coderzcolumn.com/tutorials/data-science/choropleth-maps-using-ipyleaflet

https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20List.html#selection-widgets